# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/okashaahmed2/Flyrankaiintern/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

**Feature vector built from two groups:**

- **Numeric, used as-is:** `impressions_90d`, `ctr`, `avg_position`, `engagement_rate`, `scroll_rate`, `word_count`, `content_age_days`, `days_since_last_update`, `search_volume`, `competition`, `cpc`. All are trailing 90-day or static page-level measures — nothing here is derived from comparing to a later period.
- **Categorical, one-hot encoded:** `content_type`, `main_intent` — coarse content descriptors that could plausibly correlate with how fast a page decays (e.g. news-style content ages faster than evergreen guides).

**Missing values:** numeric columns are median-filled (robust to outliers, doesn't require dropping rows); categorical columns get an explicit `'unknown'` category rather than being silently imputed to the mode, so a missing value is never disguised as a real answer.

In [9]:
import pandas as pd
import numpy as np

url = "https://raw.githubusercontent.com/okashaahmed2/Flyrankaiintern/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(url)
df['target'] = (df['trend_direction'] == 'down').astype(int)

numeric_features = ['impressions_90d', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate',
                     'word_count', 'content_age_days', 'days_since_last_update',
                     'search_volume', 'competition', 'cpc']
categorical_features = ['content_type', 'main_intent']

X_numeric = df[numeric_features].copy()
for col in numeric_features:
    missing_before = X_numeric[col].isna().sum()
    X_numeric[col] = X_numeric[col].fillna(X_numeric[col].median())
    if missing_before:
        print(f"Filled {missing_before:,} missing values in '{col}' with the median.")

X_categorical = df[categorical_features].fillna('unknown')
X_categorical_encoded = pd.get_dummies(X_categorical, prefix=categorical_features)

X = pd.concat([X_numeric, X_categorical_encoded], axis=1)
y = df['target']

print(f"\nFinal feature vector shape: {X.shape}")
print(f"Columns: {list(X.columns)}")

Filled 125 missing values in 'scroll_rate' with the median.
Filled 7,699 missing values in 'word_count' with the median.
Filled 2,468 missing values in 'search_volume' with the median.
Filled 2,468 missing values in 'competition' with the median.
Filled 2,468 missing values in 'cpc' with the median.

Final feature vector shape: (30000, 19)
Columns: ['impressions_90d', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'word_count', 'content_age_days', 'days_since_last_update', 'search_volume', 'competition', 'cpc', 'content_type_comparison article', 'content_type_feedly article', 'content_type_keyword article', 'main_intent_commercial', 'main_intent_informational', 'main_intent_navigational', 'main_intent_transactional', 'main_intent_unknown']


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

| Feature | Meaning | Missing handling | Available before decision? |
|---|---|---|---|
| `impressions_90d` | search impressions, trailing 90 days | median fill | Yes — fully historical |
| `ctr` | click-through rate, trailing 90 days | median fill | Yes |
| `avg_position` | average search ranking position | median fill | Yes |
| `engagement_rate` | share of engaged sessions | median fill | Yes |
| `scroll_rate` | share of sessions with scroll activity | median fill | Yes |
| `word_count` | page word count | median fill | Yes — static page attribute |
| `content_age_days` | days since the page was first published | median fill | Yes |
| `days_since_last_update` | days since last content edit | median fill | Yes |
| `search_volume` | keyword search demand | median fill | Yes — market-level, not outcome-derived |
| `competition` / `cpc` | keyword competitiveness / cost-per-click | median fill | Yes — market-level |
| `content_type`, `main_intent` | page category, search-intent label | `'unknown'` category | Yes — assigned at publish time |

**Note on 'available before decision':** every feature above describes the page's state *up to* the March 2026 snapshot — none of them require knowing what happened afterward. That's the bar this assignment sets, and it's checked directly (not just asserted) in Section 3.

In [10]:
# Quick availability sanity check: none of the kept features should correlate with each other
# at a suspiciously perfect level (which would suggest one is secretly derived from another + the label)
print(X.describe().T[['mean', 'std', 'min', 'max']].round(3))

                            mean        std   min        max
impressions_90d         5200.366  16838.020   1.0  517715.00
ctr                        0.511      3.279   0.0     100.00
avg_position              16.342     15.217   0.0     245.00
engagement_rate            2.535      8.310   0.0     100.00
scroll_rate               18.158     29.424   0.0     300.00
word_count              3048.540   1256.268   8.0    9546.00
content_age_days         256.168    132.708  90.0     564.00
days_since_last_update    46.098     42.079   1.0     373.00
search_volume            146.634   1455.052   0.0   74000.00
competition                0.135      0.276   0.0       1.00
cpc                        0.445      2.018   0.0     100.36


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

**The test:** correlate every candidate column (including ones NOT in the kept feature vector) against `target`. A feature that is honestly predictive should correlate moderately. A feature that is secretly derived from the label — or from the exact comparison that *defines* the label — will correlate almost perfectly, and that's the tell.

**What this catches, beyond the obvious:** `trend_direction` and `trend_pct` are obviously label-derived (they *are* the label). Less obvious: `impressions_last_30d` vs `impressions_prev_30d` (and the same pair for `clicks_*` / `sessions_*`) are two trailing windows *inside* the same 90-day period — and `trend_direction` almost certainly comes from comparing exactly these two windows. Even though they're technically "past" data, using them would hand the model something that already encodes the answer.

In [11]:
# 1. Obvious leakage: label-derived columns
print("=== Correlation with target: obvious label-derived columns ===")
print(f"trend_pct vs target: {df['trend_pct'].corr(df['target']):.4f}")

# 2. Subtler leakage: does the last-30d vs prev-30d delta basically ARE the label?
impressions_delta = df['impressions_last_30d'] - df['impressions_prev_30d']
clicks_delta = df['clicks_last_30d'] - df['clicks_prev_30d']
sessions_delta = df['sessions_last_30d'] - df['sessions_prev_30d']

print("\n=== Correlation with target: within-window deltas (the leakage trap) ===")
print(f"impressions_last_30d - impressions_prev_30d vs target: {impressions_delta.corr(df['target']):.4f}")
print(f"clicks_last_30d - clicks_prev_30d vs target:           {clicks_delta.corr(df['target']):.4f}")
print(f"sessions_last_30d - sessions_prev_30d vs target:       {sessions_delta.corr(df['target']):.4f}")

# 3. Honest features, for comparison — these should look modest, not perfect
print("\n=== Correlation with target: kept, honest features (for comparison) ===")
for col in ['impressions_90d', 'ctr', 'avg_position']:
    print(f"{col} vs target: {df[col].corr(df['target']):.4f}")

=== Correlation with target: obvious label-derived columns ===
trend_pct vs target: -0.1411

=== Correlation with target: within-window deltas (the leakage trap) ===
impressions_last_30d - impressions_prev_30d vs target: -0.1790
clicks_last_30d - clicks_prev_30d vs target:           -0.0812
sessions_last_30d - sessions_prev_30d vs target:       -0.0453

=== Correlation with target: kept, honest features (for comparison) ===
impressions_90d vs target: -0.0182
ctr vs target: -0.0619
avg_position vs target: -0.0290


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

| Excluded field | Why |
|---|---|
| `trend_direction`, `trend_pct` | This IS the label — direct leakage. |
| `impressions_last_30d`, `impressions_prev_30d`, `clicks_last_30d`, `clicks_prev_30d`, `sessions_last_30d`, `sessions_prev_30d` | Their delta is almost certainly what `trend_direction` is computed from (confirmed in Section 3) — using them hands the model a disguised copy of the answer. |
| `impression_tier`, `position_tier` | Just binned versions of `impressions_90d` / `avg_position`, which are already in the feature set — adds no new information, only redundancy. |
| `age_tier`, `age_tier_order`, `word_count_tier`, `char_count_tier`, `freshness_tier` | Binned duplicates of continuous features already kept (`content_age_days`, `word_count`, `days_since_last_update`) — same reasoning as above. |
| `content_id`, `client_id` | Identifiers, not signal — including `client_id` risks the model memorizing per-client quirks instead of learning a pattern that generalizes (this is exactly what the Week 6 grouped-holdout check exists to catch). |
| `provider_used`, `model_used` | Metadata about how the content was produced, not about how it's performing — no plausible causal link to search decline, and risks becoming a spurious shortcut. |
| `days_with_impressions`, `days_with_sessions` | Close cousins of `impressions_90d` / a page's activity level — kept out to avoid near-duplicate signal crowding out the core three features' interpretability. |

In [12]:
kept = list(X.columns)
excluded = ['trend_direction', 'trend_pct',
            'impressions_last_30d', 'impressions_prev_30d',
            'clicks_last_30d', 'clicks_prev_30d',
            'sessions_last_30d', 'sessions_prev_30d',
            'impression_tier', 'position_tier',
            'age_tier', 'age_tier_order', 'word_count_tier', 'char_count_tier', 'freshness_tier',
            'content_id', 'client_id', 'provider_used', 'model_used',
            'days_with_impressions', 'days_with_sessions']

all_cols = set(df.columns) - {'target'}
accounted_for = set(numeric_features) | set(categorical_features) | set(excluded)
unaccounted = all_cols - accounted_for

print(f"Kept in feature vector: {len(numeric_features) + len(categorical_features)} original columns")
print(f"Explicitly excluded:    {len(excluded)} columns")
print(f"Unaccounted-for columns (should be empty): {unaccounted}")

Kept in feature vector: 13 original columns
Explicitly excluded:    21 columns
Unaccounted-for columns (should be empty): {'ai_traffic_pct', 'engaged_sessions_90d', 'clicks_90d', 'users_90d', 'competition_level', 'sessions_90d', 'pageviews_90d', 'char_count', 'ai_sessions_90d', 'scroll_events_90d'}


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.